# Setup

In [ ]:
%%capture
# 1. Install dependencies
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai pandas
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [ ]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent
from pydantic import BaseModel

MODEL = OpenAIChatModel(
    # 'openai/gpt-4o-mini',
    'openai/gpt-5.4-mini',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

# Tools

A **plain LLM call** has no memory of the world after its training cutoff and no tools. An **agent** has a loop: it can call tools, see the results, and reason over them.

In [ ]:
# (a) Plain agent — no tools
from pydantic_ai import Agent

plain_agent = Agent(MODEL)
result = plain_agent.run_sync('What is today\'s date?')
print(result.output)

In [ ]:
# (b) Agent with one tool
from datetime import date

def get_today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return date.today().isoformat()

tool_agent = Agent(MODEL, tools=[get_today])

result = tool_agent.run_sync('What is today\'s date?')
print(result.output)

**What just happened?**

The first agent had no way to know the current date. The second agent had a tool — `get_today()` — and the LLM decided to call it.

Critically, the LLM doesn't execute the function itself. It returns a *tool-call request*, the framework runs the function, the result is appended to the conversation, and the LLM then produces its final answer.


In [ ]:
# Tool name, arguments, and docstring are visible to the agent, but not the implementation
from pprint import pprint
for toolset in tool_agent.toolsets:
    for name, tool in toolset.tools.items():
        print(name)
        pprint(tool.function_schema)

# Trace 

In [ ]:
print(f'Final answer: {result.output}\n')
print('Trace (each ModelMessage in the conversation):')

def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)
            print(f'    └─ {kind}: {snippet}')

pretty_print_trace(result)

# Exercise 1 - GC content

In [ ]:
import random
dna_sequence = "".join(random.choices("ATGC", weights=[0.15,0.15,0.35,0.35], k=100))
Agent(MODEL).run_sync(f"Whats the GC content of {dna_sequence}?")

In [ ]:
# Exercise 1 - Create an agent with a tool to calculate GC content and ask him to calculate it for dna_sequence
# The GC content of the sequence is the percentage of either G or C in the sequence, for example 84.5
def get_gc_content(sequence: str): #parameters must be typed
    pass

get_gc_content('GCCCTATA')

In [ ]:
gc_agent = #TODO
result = #TODO
pretty_print_trace(result)

# Exercise 2 - Data inspector

In [ ]:
# Setup - run this to create a csv with random DNA sequences
import random
import pandas as pd

def make_an_experiment_csv(num_of_samples, csv_path, seed=42):
    rng = random.Random(seed)
    sequence_lengths = [rng.randint(100, 1000) for _ in range(num_of_samples)]
    pd.DataFrame({
        "sequence": [
            "".join(rng.choices("ATCG", k=sequence_lengths[i]))
            for i in range(num_of_samples)
        ],
        "sequence_len": sequence_lengths,
        "score": [100/(rng.random()*sequence_lengths[i]) for i in range(num_of_samples)]
    }).to_csv(csv_path, index=False)

make_an_experiment_csv(2000, "experiment_1.csv", 42)
make_an_experiment_csv(1000, "experiment_2.csv", 321)
csv_names = ["experiment_1.csv", "experiment_2.csv"]

In [ ]:
# Exercise 2 - write a function that returns the stats of a CSV file
# Hint 1 - use pandas to read the csv and the describe() method to get the stats
# Hint 2 - tool return value need to be JSON-serializable - use to_dict() to convert the stats to a dictionary

import pandas as pd
def get_csv_stats(csv_path: str):
    pass

get_csv_stats('experiment_1.csv')

In [ ]:
# We need to inform the agent of our files - it has no tools to inspect the directory
csv_agent = Agent(MODEL, tools=[get_csv_stats], system_prompt=f"You have access to the following files: {csv_names}")
pretty_print_trace(csv_agent.run_sync("What is the highest score across all experiments?"))

# Exercise 3 - combining tools

In [ ]:
# we want to keep tools generic and efficient (e.g. running get_gc_content on 1000s of sequences = 1000s of tool calls)
# we can also add a docstring that will be visible to the agent
def add_gc_content_column(csv_to_read: str, sequence_column: str, new_csv_path: str):
    """
    Args:
        csv_to_read (str): The path to the CSV file to read.
        sequence_column (str): The name of the column containing the sequences.
        new_csv_path (str): The path to save the new CSV file with the GC content column
    """
    df = pd.read_csv(csv_to_read)
    df["GC_content"] = df[sequence_column].apply(get_gc_content) #Re-using your existing function
    df.to_csv(new_csv_path, index=False)

multitool_agent = Agent(MODEL, tools=[add_gc_content_column, get_csv_stats], system_prompt=f"You have access to the following files: {csv_names}")

In [ ]:
pretty_print_trace(multitool_agent.run_sync(f"Which experiment has the highest GC content sequence?"))

In [ ]:
pretty_print_trace(multitool_agent.run_sync(f"Which experiment has more sequences?")) #this is a new run, does not have previous context

In [ ]:
#TODO find a data-related question the multitool_agent can't answer or hallucinates